# Assignment 1: Evidence from recorded data

INTC, Nasdaq, 21 June 2012, LOBSTER sample at depth 10.

## 0. Setup

This part prepares the data for Assignment 1. 

I use LOBSTER data, which has two files per trading day. The first one is a message file including every order, cancellation and trade. The second one is an orderbook file, that is what the limit order book looked like after each message. 

I pick the stock INTC with 10 levels of the order book. I set the tick size (USD = 0.01) and the measures I will compute later. That is queue imbalance, sweep cost and order flow imbalance (OFI) over time windows of 60 and 300 seconds. 

Finally I chech that the two files have the same number of rows, because row k of the orderbook file belongs to message k of the message file. 

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from unito26.lob import lobster
from unito26.lob.lobster_session import LobsterMarketSession
from unito26.lob.messages import GridDepth, SweepSize, TickGrid
from unito26.lob.statistics import SessionStatistics

DATA = Path("..") / "data" / "lobster"
CENT = TickGrid(0.01)
SPEC = SessionStatistics((GridDepth(1), GridDepth(5)), (SweepSize(100),), (60,300))

PAIRS = [lobster.LobsterFiles.parse(p) for p in sorted(DATA.glob("*_message_*.csv"))]
INTC = next(f for f in PAIRS if f.ticker == "INTC" and f.reported_depth == 10)
print(f"Selected file pair: {INTC.ticker} on {INTC.day}, {INTC.reported_depth} reported levels of the book")


In [ ]:
census = lobster.file_census(PAIRS)
census.assign(Aligned=census["MessageRows"] == census["BookRows"])

The data covers a full trading day, 09:30 (34200 s) to 16:00 (57600 s). Both files have 624,040 rows, so they are aligned. 

## 1. Part i: Reconstructing the book

### 1.1 Data checks

Before reconstructing the book, I check that the message file can be trusted and used as Section 1.2.5 describes. I count the message types, build the session over regular trading hours, and check whether any visible execution took place away from the best price. Since LOBSTER records a trade as the execution of the resting order, one market order can span several rows, so I also count market orders under three conventions for grouping the executions. Finally I check every message against simple field constraints: a positive order ID, a non-negative price on the cent grid, and a positive size. The counts are compared with the documented values in `documentation/from-lobster-files-to-a-session.md`. 

In [ ]:
lobster.event_census(PAIRS).set_index(["Ticker", "Depth"])

In [ ]:
raw = LobsterMarketSession.from_files(INTC, SPEC, CENT, lobster.NASDAQ_REGULAR_HOURS)
off = raw.executions_off_the_touch()
visible = (raw.messages["Type"] == lobster.LobsterEvent.EXECUTION_VISIBLE).sum()
print(f"Visible executions: {visible}, {len(off)} of the executions are away from the best price on the resting side.")

In [ ]:
from unito26.lob.lobster_session import market_order_census

market_order_census([INTC]).set_index(["Ticker", "Convention"])

In [ ]:
UNIT = lobster.price_unit(CENT)
lobster.constraint_census(raw.messages, UNIT).set_index("Constraint")

The two files hold 624,040 rows each, so the positional alignment of Section 1.2.5 is at least consistent. Only about 5% of the messages are trades: 28,924 visible and 3,559 hidden executions. There are no cross trades and no trading halts. 

All 28,924 visible executions happened at the best price. This means no market order walked the book, so OFI is kept exactly when executions are grouped into market orders (Remark 1.2.6). The three ways of counting market orders give almost the same result, about 8,000. Only the 3,559 hidden executions break the rules: they have no order ID, and 2,140 of them have a price off the tick grid. These may cause differences when I compare the books. 

### 1.2 The two files, and the opening book

In this part I look at the first rows of the two files, to see how they fit together. Then I build the opening book from the first row of the orderbook file. This is the book I will start from when I reconstruct the book from the messages. 

In [ ]:
display(raw.messages.head(), raw.book.head())

In [ ]:
from unito26.lob.messages import ReportedDepth
from unito26.lob.orderbook import AggregateBook

DEPTH = ReportedDepth(10)
opening = AggregateBook.from_lobster_row(raw.book.iloc[0], UNIT, DEPTH)
print("Best bid in the opening book:", opening.best_bid_price, " Best ask in the opening book:", opening.best_ask_price)

Row k of the orderbook file shows the book after message k (Section 1.2.5). For example, message 0 executes 34 shares at \$27.52, and row 0 shows 66 shares left there. Message 1 adds 100 shares and row 1 shows 166. 

The book before 09:30 is not known, so the fold starts from row 0. At the start only the ten best levels on each side are known. The opening book has best bid of 2751 ticks (\$27.52), and a best ask of 2752 ticks (\$27.52), so the spread is one tick. 

### 1.3 The fold

I rebuild the book by applying each message from 1 onwards to the opening book, using `size_at` and `set_size`. A submission adds its size at its price. In LOBSTER a submission always rests, since trades are recorded as executions of the resting order (Remark 1.2.5). A partial cancellation, a deletion or a visible execution removes its size at its price, on the side given by 'Direction'. A hidden execution does not change the visible book. If a removal asks for more shares than the book holds, the size is set to zero and the event is counted. 

In [ ]:
rebuilt = opening.copy()
rest = raw.messages.iloc[1:]
rebuilt_rows = np.empty((len(rest), 4 * DEPTH), dtype="int64")
unknown = 0

for n, (kind, size, price, side) in enumerate(
        zip(rest["Type"], rest["Size"], rest["Price"], rest["Direction"])):
    ticks, size, side = int(price) // UNIT, int(size), int(side)
    resting = rebuilt.size_at(side, ticks)
    if kind == lobster.LobsterEvent.SUBMISSION:
        rebuilt.set_size(side, ticks, resting + size)
    elif kind != lobster.LobsterEvent.EXECUTION_HIDDEN:
            if size > resting: 
                 unknown += 1
            rebuilt.set_size(side, ticks, max(resting - size, 0))
    rebuilt_rows[n] = rebuilt.to_lobster_row(UNIT, DEPTH)

print(f"Rows rebuilt: {len(rebuilt_rows):,} (one for each message after message 0)")
print(f"Removals of shares the fold did not know about: {unknown:,}")

The fold gives 624,039 rows, one for each message after message 0. In 1,333 cases a message removed shares the fold did not know about. At the start only the ten best levels on each side are known, so these are most likely orders resting deeper in the book, which were cancelled or executed later. 

### 1.4 The comparison with the orderbook file

I compare the rebuilt book with the orderbook file row by row. Row n of the book and row n + 1 of the file both show the book after message n + 1. I check how often they agree in all 40 columns, at the best quotes only, and level by level. 

In [ ]:
rebuilt_book = rebuilt_rows
file_book = raw.book.iloc[1:].to_numpy(dtype="int64")

same_row = (rebuilt_book == file_book).all(axis=1)
same_touch = (rebuilt_book[:, :4] == file_book[:, :4]).all(axis=1)
print(f"Rows compared: {len(rebuilt_book):,}")
print(f"Rows identical in all 40 columns: {same_row.mean():.2%}")
print(f"Rows identical at the best quotes (level 1): {same_touch.mean():.2%}")

match = pd.DataFrame(
    (rebuilt_book == file_book).mean(axis=0).reshape(10, 4),
    index=pd.Index(range(1,11), name="Level"),
    columns=["AskPrice", "AskSize", "BidPrice", "BidSize"]
)
match.style.format("{:.2%}")

Only 0.01% of the rows agree in all 40 columns, and 2.36% at the best quotes. The two sides behave very differently. On the ask side the prices agree in about 89% of the rows at every level, and the sizes agree less the deeper the level, from 87% at level 1 to 16% at level 10. On the bid side the prices agree in only about 3.5% of the rows, even at the best quote. The next part explains these discrepancies. 

### 1.5 The discrepancies

If the orderbook file were exactly the sum of the messages, the fold could go wrong in one way. Every share it adds comes from a submission in the file, so it can never add too much. But it does not know the orders beyond the ten levels known at the start, so when such an order is removed, it sets the size to zero and keeps too little. So where the best prices agree, the rebuilt size could be smaller than the file's, but never larger. The cells below find the first discrepancy, test this, and trace where the bid side goes wrong. 

In [ ]:
cols = list(raw.book.columns)
first = np.flatnonzero(~same_row)[0]
differs = rebuilt_book[first] != file_book[first]
print(f"First differing row: {first} (message {first + 1})")
print(raw.messages.iloc[first + 1][["Time", "Type", "Size", "Price", "Direction"]])
pd.DataFrame(
    {"Rebuilt": rebuilt_book[first][differs], "LOBSTER": file_book[first][differs]},
    index=[c for c, d in zip(cols, differs) if d],
)

In [ ]:
result = []
for side in ("Ask", "Bid"):
    price, size = cols.index(f"{side}Price1"), cols.index(f"{side}Size1")
    agree = rebuilt_book[:, price] == file_book[:, price]
    gap = rebuilt_book[agree, size] - file_book[agree, size]
    result.append({
        "side": side, 
        "rows where the best price agrees": int(agree.sum()),
        "rebuilt size smaller": f"{(gap < 0).mean():.2%}",
        "equal": f"{(gap == 0).mean():.2%}",
        "rebuilt size larger": f"{(gap > 0).mean():.2%}"
    })
pd.DataFrame(result).set_index("side")

In [ ]:
bp, bs = cols.index("BidPrice1"), cols.index("BidSize1")

first_bid_price_diff = np.flatnonzero(rebuilt_book[:, bp] != file_book[:, bp])[0]
print("First row where the bid price differs:", first_bid_price_diff, "(message", first_bid_price_diff + 1, ")")
print(" rebuilt best bid before / at:", rebuilt_book[first_bid_price_diff - 1, [bp, bs]], rebuilt_book[first_bid_price_diff, [bp, bs]])
print(" LOBSTER best bid before / at:", file_book[first_bid_price_diff - 1, [bp, bs]], file_book[first_bid_price_diff, [bp, bs]])


first_bid_surplus = np.flatnonzero((rebuilt_book[:, bp] == file_book[:, bp]) & (rebuilt_book[:, bs] > file_book[:, bs]))[0]
print("First row where the rebuilt book best bid holds more:", first_bid_surplus, "(message", first_bid_surplus + 1, ")")
print(" rebuilt best bid before / at:", rebuilt_book[first_bid_surplus - 1, [bp, bs]], rebuilt_book[first_bid_surplus, [bp,bs]])
print(" LOBSTER best bid before / at:", file_book[first_bid_surplus - 1, [bp, bs]], file_book[first_bid_surplus, [bp, bs]])

display(raw.messages.iloc[[first_bid_price_diff, first_bid_price_diff + 1, first_bid_surplus, first_bid_surplus + 1]][["Time", "Type", "Price", "Direction"]])

The first discrepancy comes at message 4. A bid order is deleted, the levels below move up, and level 10 gets a price the book never knew. The rebuilt book shows the sentinel there instead. This is expected, since only ten levels are known at the start. 

The test shows the opposite of what I expected. Where the best prices agree, the book never holds fewer shares than the file, but sometimes more. That is in 2.22% of the rows on the ask side and 16.27% on the bid side. So the rebuilt book keeps shares that the file has removed. 

On the bid side this starts with two messages. At message 10,639 (09:31:20) the best bid at \$27.49 is deleted, and \$27.48 becomes the best bid. The rebuilt book has 100 shares more there than the file. At message 10,733 a trade empties \$27.48 in the file, but the rebuilt book still has those 100 shares. No message removes them, so they stay as the best bid while the file's best bid moves on. By Remark 1.2.3, only a level-by-level comparison can find an error like this. It explains why the best bid price agrees in only about 3.5% of the rows. 

### Conclusion for part i

I rebuilt the book by applying each message as LOBSTER describes it, starting from the ten levels in the first row of the orderbook file. I then compared the rebuilt book with the file. Two kinds of discrepancy appear:

1. *Orders beyond the ten known levels.* At the start the fold only knows the ten reported levels on each side. When a level empties, the file shows the next level from deeper in the book, which the fold never knew. The first difference, at message 4, is exactly this. In 1,333 cases a message removes shares that the fold did not hold. 
2. *Shares the file removes without a message.* Where the best prices agree, the rebuilt size is never smaller than the file's, but it is larger in 2.2% of the rows on the ask side and 16.3% on the bid side. A fold that applies every message can miss shares, but it cannot add them, so the orderbook file is not exactly the sum of the messages. On the bid side, 100 extra shares at \$27.48 are left in the rebuilt book at message 10,733 and are never removed. From 09:31:20 the rebuilt best bid is therefore out of step with the file. By Remark 1.2.3, a missing message leaves the fold wrong in a way that only a level-by-level comparison can show. 

The rebuilt book agrees with the file in all 40 columns in 0.01% of the rows and at the best quotes in 2.4%. The ask prices agree in about 89% of the rows at every level. I have not found which event in the underlying feed causes the extra shares. Because of these discrepancies, parts ii-v use the book in the orderbook file and not the rebuilt book, as the package itself does. 

## 2. Part ii: The order flow imbalance and the queue imbalance

The queue imbalance at the best quote is (1.1.2) with $n = 1$:
$$I^1_t = \frac{S^{b,1}_t - S^{a,1}_t}{S^{b,1}_t + S^{a,1}_t}.$$
The contribution of the $n$-th event is (1.4.1):
$$e_n = \mathbb{1}\{P^b_{T_n} \ge P^b_{T_{n-1}}\} S^{b,1}_{T_n} - \mathbb{1}\{P^b_{T_n} \le P^b_{T_{n-1}}\} S^{b,1}_{T_{n-1}} - \mathbb{1}\{P^a_{T_n} \le P^a_{T_{n-1}}\} S^{a,1}_{T_n} + \mathbb{1}\{P^a_{T_n} \ge P^a_{T_{n-1}}\} S^{a,1}_{T_{n-1}},$$
and the order flow imbalance over the window $(t-w, t]$ is (1.4.2):
$$\mathrm{OFI}_{t,w} = \sum_{n:\, t-w < T_n \le t} e_n.$$

Because of the discrepancies in part i, I use the book in the orderbook file, not the rebuilt book. I coarsen the file pair into a session, so that the fills of one market order form one row. I use the window $w = 1$ second. Finally I summarise $e_n$, $\mathrm{OFI}_{t,1}$ and $I^1_t$ in a table. 

In [ ]:
from unito26.lob import imbalance_regression as ir
from unito26.lob.messages import Window

session = raw.coarsened(True)
mid = ir.aligned_mid_price(session)
flow = ir.aligned_order_flow(session)
ofi = ir.backward_sum(flow, Window(1.0))
imbalance = np.where(
    session.stats["QueueImbalance1Covered"].to_numpy(float) > 0,
    session.stats["QueueImbalance1"].to_numpy(float), np.nan)

print(f"Rows in the orderbook file: {len(raw.book):,}")
print(f"Rows in the session after coarsening: {len(session.lobster_book):,}")

In [ ]:
pd.DataFrame({
    "e_n": flow.values,
    "OFI over 1 s": ofi,
    "I^1": imbalance,
}).describe().T

Coarsening reduces the 624,040 rows of the file to 599,592. The 28,924 visible fills become 8,035 market orders, and the 3,559 hidden executions are absorbed or dropped. 

The contribution $e_n$ is zero at the median. Many events do not change the best prices or the sizes there, and they contribute nothing. The extremes are large, about ±82,000 shares, because an event that moves a best price contributes a whole queue. Summed over one second $\mathrm{OFI}_{t,1}$ varies far more than $e_n$, with a standard deviation of about 27,500 shares. OFI is missing for the first second of the session, where the window $(t-1, t]$ is not yet complete. 

$I^1_t$ stays in $[-1, 1]$, as it must, and is slightly positive on average (0.064). So over the day the bid queue was on average a little larger than the ask queue. 

### Conclusion for part ii
$\mathrm{OFI}_{t,1}$ is computed by Definition 1.4.2 and $I^1_t$ by (1.1.2) on the 599,592 rows of the coarsened session. 

## 3. Part iii: The mid-price change regressed on OFI

By Proportion 1.4.3, 
$$\Delta P^m_{t,w} = P^m_t - P^m_{t-w} = \frac{\tau}{2\bar S}\,\mathrm{OFI}_{t,w} + \varepsilon_{t,w}, \qquad |\varepsilon_{t,w}| \le \tau,$$
with the residual (1.4.4)
$$\varepsilon_{t,w} = -\frac{\tau}{2\bar S}\Big[\big(S^{b,1}_t - S^{b,1}_{t-w}\big) - \big(S^{a,1}_t - S^{a,1}_{t-w}\big)\Big].$$
With prices in ticks, $\tau = 1$, so a regression slope $\hat\beta$ implies the depth $\bar S = 1/(2\hat\beta)$.

To see how much of the difference the residual accounts for, the same regression is run on
$\mathrm{OFI}_{t,w} - (\Delta S^b - \Delta S^a)$, which under Assumption 1.4.1 equals
$2\bar S\,\Delta P^m_{t,w}$ exactly. The comparison is made for w = 1, 10 and 60 seconds.

In [ ]:
from scipy import stats

book = session.lobster_book
bid_size = ir.AlignedSeries(mid.times, book["BidSize1"].to_numpy(float), mid.segments)
ask_size = ir.AlignedSeries(mid.times, book["AskSize1"].to_numpy(float), mid.segments)

rows = []
for w in (1.0, 10.0, 60.0):
    mid_change = ir.backward_change(mid, Window(w))
    ofi_w = ir.backward_sum(flow, Window(w))
    queues = ir.backward_change(bid_size, Window(w)) - ir.backward_change(ask_size, Window(w))
    ok = np.isfinite(mid_change) & np.isfinite(ofi_w) & np.isfinite(queues)
    fit = stats.linregress(ofi_w[ok], mid_change[ok])
    fit_residual = stats.linregress(ofi_w[ok] - queues[ok], mid_change[ok])
    rows.append({
        "w (s)": w,
        "slope (ticks per share)": fit.slope,
        "implied depth (shares)": 1 / (2 * fit.slope),
        "R^2 with OFI": fit.rvalue ** 2,
        "R^2 with OFI - (ΔS^b - ΔS^a)": fit_residual.rvalue ** 2
    })

print(f"Median size at the best quotes: {np.median(np.r_[book["BidSize1"], book["AskSize1"]]):,.0f} shares")
pd.DataFrame(rows).set_index("w (s)")

The slope is stable across the three windows, about $2.1 \times 10^{-5}$ ticks per share, or $2.1 \times 10^{-7}$ dollars per share after multiplying by $\tau = \$0.01$. It implies a depth $\bar S$ of 24,000-25,000 shares, about twice the median size at the best quotes (12,847 shares). By Remark 1.4.5, the residual is the part of the flow that went into the queues instead of moving the prices. So some OFI comes with no change in the mid-price. The regression then sees large OFI with small price changes, which gives a smaller slope and a larger implied depth. 

OFI alone explains 55-58% of the variance of the mid-price change. Adding the residual (1.4.4) raises $R^2$ by 3-5 percentage points, to 59-63%. Under Assumption 1.4.1 the second regression would be exact, so the residual accounts for only a small part of the difference. The rest comes from the ways the data breaks the assumption. The queues are not all of size $\bar S$ (Assumption 1.4.1 iii.), and events also happen behind the best quotes (Assumption 1.4.1 i.).

### Conclusion for part iii

The regression implies a depth of about 24,000-25,000 shares, around twice the median size at the best quotes. The residual (1.4.4) explains only 3-5 percentage points more of the variance, so most of the difference comes from the data breaking Assumption 1.4.1.

## 4. Part iv: Does OFI forecast the mid-price?

I measure the forecasting relation by the correlation
$$\rho(w, h) = \mathrm{corr}\big(\mathrm{OFI}_{t,w},\; P^m_{t+h} - P^m_t\big)$$
over a grid of windows $w$ and horizons $h$. To keep the observations from overlapping, I sample $t$ once every $\max(w, h)$ seconds. With $n$ such observations, a correlation is marked readable (*) when $|\rho| >2/\sqrt{n}$. This is roughly the 95% band for a correlation of zero between independent observations, so a smaller correlation could easily come from chance. 

In [ ]:
from unito26.lob.messages import Horizon

times = np.asarray(mid.times, dtype=float)

def relation(signal, h, spacing):
    ahead = ir.forward_change(mid, Horizon(h))
    grid = np.arange(times[0] + spacing, times[-1] - h, spacing)
    rows = np.searchsorted(times, grid, side="right") -1
    x, y = signal[rows], ahead[rows]
    ok = np.isfinite(x) & np.isfinite(y)
    corr = np.corrcoef(x[ok], y[ok])[0, 1]
    return corr, 2 / np.sqrt(ok.sum())

windows = (1.0, 10.0, 60.0, 300.0)
horizons = (1.0, 10.0, 60.0, 300.0)
table = {}
for w in windows:
    ofi_w = ir.backward_sum(flow, Window(w))
    for h in horizons:
        corr, noise = relation(ofi_w, h, max(w, h))
        table[(w, h)] = f"{corr:+.3f}" + (" *" if abs(corr) > noise else "")

pd.Series(table).unstack().rename_axis(index="w (s)", columns="h (s)")

The relation can only be read at short windows and horizons. For $w$ and $h$ of 1 or 10 seconds the correlation is positive, between +0.047 and +0.072, and exceeds $2/\sqrt{n}$. So when the order flow has been buy-heavy, the mid-price tends to rise a little in the next seconds. The relation is weak because a correlation of 0.07 explains under 1% of the variance, against 55-58% in part iii. 

At 60 seconds or more, no correlation exceeds the band, and the signs vary. One reason is the small sample: at 60 and 300 seconds there are only about 390 and 78 observations, so the band widens to about 0.10 and 0.23. By Remark 1.4.24, the forecast is also expected to fade at long horizons. One day of data cannot show whether the relation disappears there, only that it cannot be read.

### Conclusion for part iv

OFI forecasts the mid-price with a positive, but weak correlation, and only for windows and horizons of 1-10 seconds. At 60 seconds or more the relation cannot be read. 

## 5. Part v: Does the queue imbalance forecast the mid-price?

I repeat the test of part iv with $I^1_t$ in place of $\mathrm{OFI}_{t,w}$. $I^1_t$ is measured at one instant and not summed over a window, so only the horizon $h$ varies. To compare the two fairly, I measure OFI over $w = 1$ seconds in the same way beside it, sampling once every $h$ seconds. 

In [ ]:
signals = {"I^1": imbalance, "OFI, w = 1 s": ir.backward_sum(flow, Window(1.0))}
table = {}
for name, signal in signals.items():
    for h in horizons:
        corr, noise = relation(signal, h, h)
        table[(name, h)] = f"{corr:+.3f}" + (" *" if abs(corr) > noise else "")

pd.Series(table).unstack().rename_axis(index="signal", columns="h (s)")

The OFI row is the same as the first row in part iv, so the signals are measured in the same way. 

$I^1_t$ forecasts the mid-price much better than OFI at short horizons. The correlation is +0.170 at $h = 1$ seconds and +0.230 at $h = 10$ seconds, about three to four times that of OFI, and both exceed the band. The sign is positive because when the bid queue is larger than the ask queue, the mid-price tends to rise. At 60 and 300 seconds the sign is still positive, but neither value exceeds the band. 

By Remark 1.5.1, the comparison in generated data is asymmetric. There is no informed trading there, so $I^1_t$ can only forecast mechanically. A small queue is one market order from being cleared, and clearing it moves the mid-price. In recorded data the flow is traded, and Section 1.5 explains that a thin bid can be thin because those posting on that side expected the price to fall. So $I^1_t$ can also carry information, and the comparison is symmetric. The result can then be read as it stands, and here it favours $I^1_t$ in a real market. 

The test has two limits. It does not separate the mechanical part of $I^1_t$ from the informative part, and it compares the two signals one at a time, not what each adds to the other. 

### Conclusion for part v

The queue imbalance $I^1_t$ forecasts the mid-price better than OFI at horizons of 1-10 seconds, with correlations of +0.17 to +0.23. Since the flow is traded, the comparison is symmetric, so this result favours $I^1_t$ in a real market. 

## 6. Conclusion

The rebuilt book does not match the orderbook file exactly. Some orders lie beyond the ten known levels, and the file removes shares that no message removes. Parts ii-v therefore use the book in the orderbook file. 

OFI accounts for much of the mid-price change over the same window: it explains 55-58% of the variance, with a slope that implies a depth of about 24,000-25,000 shares. The residual adds only 3-5 percentage points, so most of the rest comes from the data breaking Assumption 1.4.1.

OFI forecasts much less. The correlation with the future mid-price change is positive but weak, and can only be read for windows and horizons of 1-10 seconds. The queue imbalance $I^1_t$ forecasts better at the same horizons, with correlations three to four times larger. 

All results come from one ticker on one trading day, so they show what can be read in this data, not whether they hold in general. 